# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook guides you through loading and exploring the FAIR² dataset using the `mlcroissant` library. All record sets, fields, and columns are referenced by their global `@id`s according to the Croissant schema.

### Dataset Source
The dataset source is provided via the following Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install --quiet mlcroissant

## 1. Data Loading
Load metadata and records from the FAIR² dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load Dataset metadata
dataset = mlc.Dataset(croissant_url)

# Print out main metadata fields
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}\n")
print(f"Version: {metadata.version}")
print(f"Identifier: {metadata.identifier}")
print(f"Published: {metadata.datePublished}")
print(f"License: {metadata.license}")
print(f"Personal sensitive fields: {getattr(metadata, 'personalSensitiveInformation', None)}")

## 2. Data Overview

Let's review the available record sets and their fields by inspecting the dataset's Croissant structure. All references use `@id` values following the Croissant specification.

**Note:** The list of record sets and field `@id`s may be printed for reference.

In [ ]:
# List all record sets and their fields in this dataset.
print("Available record sets and their fields (by @id):")

record_sets = list(dataset.record_sets)
if not record_sets:
    print("No explicit record sets found; using main CSV record set.")
    # For most tabular croissant datasets, there's a single main record set;
    # we can discover it by listing available record sets after loading the dataset.
    # mlcroissant provides 'record_sets' as an iterator of mlcroissant.RecordSet objects
    record_sets = list(dataset.record_sets)
    if not record_sets:
        print("Dataset has no defined record sets.")
    else:
        for rs in record_sets:
            print(f"  Record set @id: {rs.id}")
            print("    Fields:")
            for field in rs.fields:
                print(f"      Field: {field.id}")
else:
    for rs in record_sets:
        print(f"  Record set @id: {rs.id}")
        print("    Fields:")
        for field in rs.fields:
            print(f"      Field: {field.id} ({field.name})")

For convenience, let's print a few sample records for the primary record set. All references are by the record set's “@id”, as above.

In [ ]:
# Get the primary record set @id (should be unique)
primary_record_set = None
for rs in dataset.record_sets:
    primary_record_set = rs.id
    break

print(f"Displaying a few sample records from record set: {primary_record_set}")
n = 3
for idx, rec in enumerate(dataset.records(record_set=primary_record_set)):
    print(f"Record #{idx+1}:")
    for k, v in rec.items():
        print(f"  {k}: {v}")
    if idx + 1 >= n:
        break

## 3. Data Extraction
Load data from the chosen record set (referenced by its Croissant `@id`) into a `pandas` DataFrame for analysis. All field/column names are referenced using their `@id` from the Croissant schema.

In [ ]:
# Gather all available record set @ids (should be only one for this tabular dataset)
record_set_ids = [rs.id for rs in dataset.record_sets]

dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)

# Print columns of the main record set DataFrame
main_rs_id = record_set_ids[0]
print(f"Columns in DataFrame for record set '{main_rs_id}':\n", dataframes[main_rs_id].columns.tolist())

# Show the first few rows
dataframes[main_rs_id].head()

## 4. Exploratory Data Analysis (EDA)

Let's apply some initial data processing steps. 
- We'll select a numeric field (e.g., **age at second cancer diagnosis**) referenced by its `@id` in the Croissant schema. 
- We'll filter for records with age above a threshold and normalize age.
- Then we'll group by a categorical field (e.g., **sex**) using its Croissant `@id`.

**Note:** Make sure the field `@id`s correspond exactly to those read from the DataFrame previously.

In [ ]:
# Let's inspect DataFrame columns to pick target fields (all are by Croissant @id)
print("Sample columns (Croissant @id):", dataframes[main_rs_id].columns.tolist())

# Based on typical clinical data, we'll look for an age field and a sex field by their Croissant @id (here, we must inspect and use the actual field IDs)
# Suppose age field is '@id': 'https://api.app.sen.science/frontiers/7862866/field-age_at_2nd_diagnosis' and sex field is '@id': 'https://api.app.sen.science/frontiers/7862866/field-sex'
# If their exact IDs cannot be guessed, print the column names first. For this example, we select likely-named Croissant @ids:

numeric_field_id = None
sex_field_id = None
for col in dataframes[main_rs_id].columns:
    if 'age' in col.lower():
        numeric_field_id = col
    if 'sex' in col.lower() or 'gender' in col.lower():
        sex_field_id = col

print(f"Selected numeric field Croissant @id: {numeric_field_id}")
print(f"Selected sex/group field Croissant @id: {sex_field_id}")

if numeric_field_id is not None:
    # Coerce numeric field to numeric dtype (in case it comes as string)
    dataframes[main_rs_id][numeric_field_id] = pd.to_numeric(dataframes[main_rs_id][numeric_field_id], errors='coerce')
    threshold = 55
    filtered_df = dataframes[main_rs_id][dataframes[main_rs_id][numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold}: ({len(filtered_df)} rows)")
    print(filtered_df[[numeric_field_id]].head())
    
    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} (first rows):")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by sex or equivalent categorical field, if present
    if sex_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(sex_field_id)[numeric_field_id].mean().to_frame('mean_' + numeric_field_id)
        print(f"\nGrouped by {sex_field_id} (mean {numeric_field_id}):")
        print(grouped_df)
else:
    print("No suitable numeric field found for EDA.")

## 5. Visualization

Visualize the distribution of the selected numeric field and its breakdown by the group field (using Croissant `@id` references for axis labels).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id is not None and len(filtered_df) > 0:
    # Histogram of age at 2nd diagnosis
    plt.figure(figsize=(7, 4))
    sns.histplot(data=filtered_df, x=numeric_field_id, bins=10, kde=True)
    plt.title(f"Distribution of {numeric_field_id} (> {threshold})")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()
    
    # Boxplot by sex if available
    if sex_field_id in filtered_df.columns:
        plt.figure(figsize=(7, 4))
        sns.boxplot(data=filtered_df, x=sex_field_id, y=numeric_field_id)
        plt.title(f"{numeric_field_id} by {sex_field_id}")
        plt.xlabel(sex_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()

## 6. Conclusion

In this notebook:
- We loaded the FAIR² clinical cancer dataset using the `mlcroissant` library with full Croissant `@id` referencing.
- We overviewed available record sets and fields, then extracted the main table of clinical records.
- We performed exploratory data analysis with basic statistics and grouped summaries, and visualized the data distribution.

**Next steps:** You can now conduct deeper statistical, machine learning, or clinical analyses using this structured data. For further guidance, refer to the [FAIR² dataset documentation](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json) and the [mlcroissant documentation](https://mlcommons.github.io/croissant/).